In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import pandas as pd
import numpy as np

In [12]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

In [13]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
    # "Attribution",
    # "Attribution, ID",
    # "Attribution, ID, Location",
    # "Attribution, ID, Name",
    # "Attribution, ID, Name, Location",
    # "Attribution, ID, Missing Data",
    # "Attribution, ID, Name, Missing Data",
    # "Attribution, ID, Name, Location, Missing Data",
    # "Attribution, ID, Location, Missing Data",
    # "Attribution, Location",
    # "Attribution, Missing Data",
    "ID, Location",
    "ID, Missing Data",
    "ID, Name",
    "ID, Name, Location"

]

df = df[
    df["ISSUE"].str.strip().isin(wanted_issues) &
    (df["Network ID"] != 11)
]

len(df)

df =  df[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")
df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")
df


,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,"ID, Location",3282,9,BC-Air,E278829 -> 233,Prince George Marsulex Met,"122.6672222 W, 53.845 N, Elev. 535 m -> 122.69..."
1,"ID, Location",1573,9,BC-Air,E237431 -> 344,Merritt Granite-Garcia Mobile,"120.7888889 W, 50.11277778 N, Elev. 500 m -> 1..."
2,"ID, Location",2603,9,BC-Air,0450270 -> 136,Prince George Gladstone School,"122.7613889 W, 53.85805556 N, Elev. 583 m -> 1..."
3,"ID, Location",2649,9,BC-Air,E231838 -> 327,Prince Rupert Galloway Rapids,"130.2680556 W, 54.25972222 N, Elev. 1 m -> 130..."
4,"ID, Missing Data",2615,9,BC-Air,M114217 -> 350,Castlegar Met,"117.6577778 W, 49.31388889 N, Elev. 451 m"
5,"ID, Name",13010,9,BC-Air,E290310 -> 585,-> ELKFORD ROCKY MOUNTAIN SCHOOL,"114.93342 W, 50.0077944 N, Elev. null m"
6,"ID, Name",12947,9,BC-Air,E299830 -> 469,-> Fort St John Key Learning Centre,"120.856111 W, 56.244722 N, Elev. null m"
7,"ID, Name",12257,9,BC-Air,E288469 -> 425,Prince Rupert Roosevelt Park School Met_60 -> ...,"130.3269 W, 54.3064 N, Elev. 80 m"
8,"ID, Name",12310,9,BC-Air,E234293 -> 209,Valemount Firehall -> Valemount,"119.269821 W, 52.832459 N, Elev. 800 m"
9,"ID, Name",1565,9,BC-Air,0260011 ->196,Warfield -> Warfield Elementary,"117.7469444 W, 49.09527778 N, Elev. 604 m"


In [14]:
### Split ID

# Split on '->'
split_ids = df['Native ID'].astype(str).str.split('->', expand=True)

df['old_native_id'] = split_ids[0].str.strip()
df['new_native_id'] = split_ids[1].str.strip()

df = df.drop(columns=['Native ID'])
df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")

df

,ISSUE,Station ID,Network ID,Network Name,Unique Names,Unique Location (String),old_native_id,new_native_id
0,"ID, Location",3282,9,BC-Air,Prince George Marsulex Met,"122.6672222 W, 53.845 N, Elev. 535 m -> 122.69...",E278829,233
1,"ID, Location",1573,9,BC-Air,Merritt Granite-Garcia Mobile,"120.7888889 W, 50.11277778 N, Elev. 500 m -> 1...",E237431,344
2,"ID, Location",2603,9,BC-Air,Prince George Gladstone School,"122.7613889 W, 53.85805556 N, Elev. 583 m -> 1...",0450270,136
3,"ID, Location",2649,9,BC-Air,Prince Rupert Galloway Rapids,"130.2680556 W, 54.25972222 N, Elev. 1 m -> 130...",E231838,327
4,"ID, Missing Data",2615,9,BC-Air,Castlegar Met,"117.6577778 W, 49.31388889 N, Elev. 451 m",M114217,350
5,"ID, Name",13010,9,BC-Air,-> ELKFORD ROCKY MOUNTAIN SCHOOL,"114.93342 W, 50.0077944 N, Elev. null m",E290310,585
6,"ID, Name",12947,9,BC-Air,-> Fort St John Key Learning Centre,"120.856111 W, 56.244722 N, Elev. null m",E299830,469
7,"ID, Name",12257,9,BC-Air,Prince Rupert Roosevelt Park School Met_60 -> ...,"130.3269 W, 54.3064 N, Elev. 80 m",E288469,425
8,"ID, Name",12310,9,BC-Air,Valemount Firehall -> Valemount,"119.269821 W, 52.832459 N, Elev. 800 m",E234293,209
9,"ID, Name",1565,9,BC-Air,Warfield -> Warfield Elementary,"117.7469444 W, 49.09527778 N, Elev. 604 m",0260011,196


In [15]:
#### Split Name

def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df[['old_station_name', 'new_station_name', 'n_names']] = (
    df['Unique Names'].apply(split_station_name)
)

df = df.drop(columns= 'Unique Names')

In [16]:
df

,ISSUE,Station ID,Network ID,Network Name,Unique Location (String),old_native_id,new_native_id,old_station_name,new_station_name,n_names
0,"ID, Location",3282,9,BC-Air,"122.6672222 W, 53.845 N, Elev. 535 m -> 122.69...",E278829,233,Prince George Marsulex Met,Prince George Marsulex Met,1
1,"ID, Location",1573,9,BC-Air,"120.7888889 W, 50.11277778 N, Elev. 500 m -> 1...",E237431,344,Merritt Granite-Garcia Mobile,Merritt Granite-Garcia Mobile,1
2,"ID, Location",2603,9,BC-Air,"122.7613889 W, 53.85805556 N, Elev. 583 m -> 1...",0450270,136,Prince George Gladstone School,Prince George Gladstone School,1
3,"ID, Location",2649,9,BC-Air,"130.2680556 W, 54.25972222 N, Elev. 1 m -> 130...",E231838,327,Prince Rupert Galloway Rapids,Prince Rupert Galloway Rapids,1
4,"ID, Missing Data",2615,9,BC-Air,"117.6577778 W, 49.31388889 N, Elev. 451 m",M114217,350,Castlegar Met,Castlegar Met,1
5,"ID, Name",13010,9,BC-Air,"114.93342 W, 50.0077944 N, Elev. null m",E290310,585,,ELKFORD ROCKY MOUNTAIN SCHOOL,2
6,"ID, Name",12947,9,BC-Air,"120.856111 W, 56.244722 N, Elev. null m",E299830,469,,Fort St John Key Learning Centre,2
7,"ID, Name",12257,9,BC-Air,"130.3269 W, 54.3064 N, Elev. 80 m",E288469,425,Prince Rupert Roosevelt Park School Met_60,Prince Rupert Roosevelt Park School,2
8,"ID, Name",12310,9,BC-Air,"119.269821 W, 52.832459 N, Elev. 800 m",E234293,209,Valemount Firehall,Valemount,2
9,"ID, Name",1565,9,BC-Air,"117.7469444 W, 49.09527778 N, Elev. 604 m",0260011,196,Warfield,Warfield Elementary,2


In [17]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df[cols] = df['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)

# 
df = df.drop(columns=['Unique Location (String)'])

In [18]:
df

,ISSUE,Station ID,Network ID,Network Name,old_native_id,new_native_id,old_station_name,new_station_name,n_names,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
0,"ID, Location",3282,9,BC-Air,E278829,233,Prince George Marsulex Met,Prince George Marsulex Met,1,53.845000,-122.667222,535.0,53.967500,-122.690800,577.0
1,"ID, Location",1573,9,BC-Air,E237431,344,Merritt Granite-Garcia Mobile,Merritt Granite-Garcia Mobile,1,50.112778,-120.788889,500.0,50.112800,-121.788900,500.0
2,"ID, Location",2603,9,BC-Air,0450270,136,Prince George Gladstone School,Prince George Gladstone School,1,53.858056,-122.761389,583.0,53.859200,-122.763300,583.0
3,"ID, Location",2649,9,BC-Air,E231838,327,Prince Rupert Galloway Rapids,Prince Rupert Galloway Rapids,1,54.259722,-130.268056,1.0,54.232200,-130.288900,1.0
4,"ID, Missing Data",2615,9,BC-Air,M114217,350,Castlegar Met,Castlegar Met,1,49.313889,-117.657778,451.0,49.313889,-117.657778,451.0
5,"ID, Name",13010,9,BC-Air,E290310,585,,ELKFORD ROCKY MOUNTAIN SCHOOL,2,50.007794,-114.933420,NaN,50.007794,-114.933420,NaN
6,"ID, Name",12947,9,BC-Air,E299830,469,,Fort St John Key Learning Centre,2,56.244722,-120.856111,NaN,56.244722,-120.856111,NaN
7,"ID, Name",12257,9,BC-Air,E288469,425,Prince Rupert Roosevelt Park School Met_60,Prince Rupert Roosevelt Park School,2,54.306400,-130.326900,80.0,54.306400,-130.326900,80.0
8,"ID, Name",12310,9,BC-Air,E234293,209,Valemount Firehall,Valemount,2,52.832459,-119.269821,800.0,52.832459,-119.269821,800.0
9,"ID, Name",1565,9,BC-Air,0260011,196,Warfield,Warfield Elementary,2,49.095278,-117.746944,604.0,49.095278,-117.746944,604.0


### Check and update ID

In [10]:

check_sql = sa.text("""
SELECT station_id, native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.connect() as conn:
    for _, row in df.iterrows():
        res = conn.execute(
            check_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_native_id = res.native_id

        if db_native_id == row["old_native_id"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_native_id} → {row['new_native_id']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_native_id}, expected={row['old_native_id']}"
            )

✅ OK to update station 3282: E278829 → 233
✅ OK to update station 1573: E237431 → 344
✅ OK to update station 2603: 0450270 → 136
✅ OK to update station 2649: E231838 → 327
✅ OK to update station 2615: M114217 → 350
✅ OK to update station 13010: E290310 → 585
✅ OK to update station 12947: E299830 → 469
✅ OK to update station 12257: E288469 → 425
✅ OK to update station 12310: E234293 → 209
✅ OK to update station 1565: 0260011 → 196
✅ OK to update station 12454: E316070 → 564
✅ OK to update station 12950: E221801 → 294
✅ OK to update station 13025: E320051 → 576
✅ OK to update station 13006: E326405 → 583
✅ OK to update station 13005: E326011 → 582
✅ OK to update station 12948: 0260012 → 192
✅ OK to update station 13087: E330194 → 587
✅ OK to update station 13088: E330211 → 579
✅ OK to update station 13037: E328532 → 590


In [11]:
update_station_sql = sa.text("""
UPDATE meta_station
SET native_id = :new_native_id
WHERE station_id = :station_id
  AND native_id = :old_native_id
""")

with engine.begin() as conn:
    for _, row in df.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "station_id": int(row["Station ID"]),
                "old_native_id": row["old_native_id"],
                "new_native_id": row["new_native_id"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['Station ID']}: "
                f"{row['old_native_id']} → {row['new_native_id']}"
            )
        else:
            # check why it failed
            res = conn.execute(
                check_sql,
                {"station_id": int(row["Station ID"])}
            ).fetchone()

            if res is None:
                print(f"❌ Station {row['Station ID']} not found")
            else:
                print(
                    f"⚠️ Skipped station {row['Station ID']}: "
                    f"DB={res.native_id}, expected={row['old_native_id']}"
                )

✅ Updated station 3282: E278829 → 233
✅ Updated station 1573: E237431 → 344
✅ Updated station 2603: 0450270 → 136
✅ Updated station 2649: E231838 → 327
✅ Updated station 2615: M114217 → 350
✅ Updated station 13010: E290310 → 585
✅ Updated station 12947: E299830 → 469
✅ Updated station 12257: E288469 → 425
✅ Updated station 12310: E234293 → 209
✅ Updated station 1565: 0260011 → 196
✅ Updated station 12454: E316070 → 564
✅ Updated station 12950: E221801 → 294
✅ Updated station 13025: E320051 → 576
✅ Updated station 13006: E326405 → 583
✅ Updated station 13005: E326011 → 582
✅ Updated station 12948: 0260012 → 192
✅ Updated station 13087: E330194 → 587
✅ Updated station 13088: E330211 → 579
✅ Updated station 13037: E328532 → 590


### Check and update name and geom

In [20]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_station_name = row["old_station_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_station_name = row["new_station_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ) and db_row.station_name != old_station_name:
            print(
                f"⚠️ Station {station_id} skipped (DB, XLS values differ)\n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: name = {old_station_name}, lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )

        else:
            print(
                f"✅ Station {station_id}, all match \n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}"

            )


✅ Station 3282, all match 
   DB : name = Prince George Marsulex Met, lat=NaN, lon=NaN, elev=NaN
✅ Station 1573, all match 
   DB : name = Merritt Granite-Garcia Mobile, lat=NaN, lon=NaN, elev=NaN
✅ Station 2603, all match 
   DB : name = Prince George Gladstone School, lat=53.8592, lon=-122.7633, elev=583.0
✅ Station 2649, all match 
   DB : name = Prince Rupert Galloway Rapids, lat=54.2322, lon=-130.2889, elev=1.0
✅ Station 2615, all match 
   DB : name = Castlegar Met, lat=49.31388889, lon=-117.6577778, elev=451.0
✅ Station 13010, all match 
   DB : name = ELKFORD ROCKY MOUNTAIN SCHOOL, lat=50.0077944, lon=-114.93342, elev=NaN
✅ Station 12947, all match 
   DB : name = Fort St John Key Learning Centre, lat=56.244722, lon=-120.856111, elev=NaN
✅ Station 12257, all match 
   DB : name = Prince Rupert Roosevelt Park School, lat=54.3064, lon=-130.3269, elev=80.0
✅ Station 12310, all match 
   DB : name = Valemount, lat=52.832459, lon=-119.269821, elev=800.0
✅ Station 1565, all match 
  

In [21]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_station_name,
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_station_name = row["old_station_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_station_name = row["new_station_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_station_name": new_station_name,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_station_name}, {old_lat}, {old_lon}, {old_elev}) → "
                f"({new_station_name}, {new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")


✅ Updated station 3282: (Prince George Marsulex Met, 53.845, -122.6672222, 535.0) → (Prince George Marsulex Met, 53.9675, -122.6908, 577.0)
✅ Updated station 1573: (Merritt Granite-Garcia Mobile, 50.11277778, -120.7888889, 500.0) → (Merritt Granite-Garcia Mobile, 50.1128, -121.7889, 500.0)
✅ Updated station 2603: (Prince George Gladstone School, 53.85805556, -122.7613889, 583.0) → (Prince George Gladstone School, 53.8592, -122.7633, 583.0)
✅ Updated station 2649: (Prince Rupert Galloway Rapids, 54.25972222, -130.2680556, 1.0) → (Prince Rupert Galloway Rapids, 54.2322, -130.2889, 1.0)
✅ Updated station 2615: (Castlegar Met, 49.31388889, -117.6577778, 451.0) → (Castlegar Met, 49.31388889, -117.6577778, 451.0)
✅ Updated station 13010: (, 50.0077944, -114.93342, nan) → (ELKFORD ROCKY MOUNTAIN SCHOOL, 50.0077944, -114.93342, nan)
✅ Updated station 12947: (, 56.244722, -120.856111, nan) → (Fort St John Key Learning Centre, 56.244722, -120.856111, nan)
✅ Updated station 12257: (Prince Rupert 